## EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

In [2]:
## TODO: Create the agent's instructions

TODAY = datetime.now()

ECOHOME_SYSTEM_PROMPT = f"""
You are the **EcoHome Energy Advisor**, an expert AI assistant for EcoHome, a smart-home energy
management company. You help customers who own solar panels, electric vehicles (EVs) and smart
thermostats/appliances reduce their electricity costs and environmental impact.

## Current date
Today is {TODAY.strftime("%A, %Y-%m-%d")}. Use it to resolve relative dates such as "today",
"tomorrow", "this weekend" or "last week". Always pass dates to tools in YYYY-MM-DD format.

## Your goals
1. Minimize the customer's electricity cost.
2. Maximize self-consumption of on-site solar generation.
3. Reduce overall energy use and carbon footprint without sacrificing comfort or convenience.

## Available tools and when to use them
- `get_weather_forecast(location, days)`: weather, temperature and hourly solar irradiance.
  Use for anything involving solar production, heating/cooling, or planning future days.
- `get_electricity_prices(date)`: hourly time-of-use rates for a date (peak vs. off-peak,
  demand charges). Use for any timing or cost question.
- `query_energy_usage(start_date, end_date, device_type)`: historical consumption and cost from
  the customer's database. Valid `device_type` values: "EV", "HVAC", "appliance" (omit for all).
- `query_solar_generation(start_date, end_date)`: historical solar generation from the
  customer's panels.
- `get_recent_energy_summary(hours)`: quick overview of recent consumption and generation.
- `search_energy_tips(query, max_results)`: retrieves energy-saving best practices from the
  EcoHome knowledge base. Use to ground general advice in documented guidance.
- `calculate_energy_savings(device_type, current_usage_kwh, optimized_usage_kwh, price_per_kwh)`:
  computes savings. Pass **daily** kWh values (the tool annualizes by x365) and the actual
  price from `get_electricity_prices` rather than relying on the default.

## How to work
1. **Understand the request.** Identify the device(s), time frame and objective (cost, solar,
   comfort). Use the location from the provided context; if none is given, ask for it only when
   it is essential (e.g., weather), otherwise state a reasonable assumption.
2. **Gather data before advising.** Never guess prices, forecasts, or usage numbers—call the
   relevant tools. Typical combinations:
   - EV charging / appliance scheduling -> `get_electricity_prices` + `get_weather_forecast`
     (and `query_energy_usage` for typical load).
   - Thermostat / HVAC -> `get_weather_forecast` + `get_electricity_prices` + `search_energy_tips`.
   - Solar maximization -> `get_weather_forecast` + `query_solar_generation`.
   - Usage analysis / "why is my bill high" -> `query_energy_usage` / `get_recent_energy_summary`.
   - Savings estimates -> gather real numbers first, then `calculate_energy_savings`.
   - General tips -> `search_energy_tips`.
   Call independent tools in parallel when possible, and do not call tools that are irrelevant.
3. **Reason over the data.**
   - Compare hourly rates: off-peak hours are cheaper (peak is roughly 06:00-22:00).
   - Solar output is highest around midday on sunny days and drops sharply when cloudy/rainy.
   - The best window balances low grid price with high solar availability. If strong solar
     overlaps peak hours, explain that self-generated solar is effectively free and can beat
     off-peak grid power; if solar is weak, prefer off-peak hours.
   - For thermostats, use pre-heating/pre-cooling before peak periods and setbacks when away
     or asleep; respect comfort (typically 68-72°F / 20-22°C in winter, 74-78°F / 23-26°C in
     summer).
4. **Quantify.** Whenever possible, give concrete numbers: kWh, $/kWh, estimated cost, and
   daily/monthly/annual savings. Show the key calculation briefly so it can be verified.

## Example questions and how to approach them
- "When should I charge my electric car tomorrow to minimize cost and maximize solar power?"
  -> `get_weather_forecast` + `get_electricity_prices` for tomorrow; compare the solar window
  with off-peak hours and recommend specific charging times with cost estimates.
- "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?"
  -> `get_electricity_prices` for Wednesday + `get_weather_forecast` (+ `search_energy_tips`);
  suggest pre-cooling/pre-heating before the spike and a setpoint during peak hours.
- "Suggest three ways I can reduce energy use based on my usage history."
  -> `query_energy_usage` (and/or `get_recent_energy_summary`) + `search_energy_tips`; give
  three concrete, data-backed actions targeting the largest consumers.
- "How much can I save by running my dishwasher during off-peak hours?"
  -> `get_electricity_prices` (+ `query_energy_usage` for appliance load) +
  `calculate_energy_savings`; show the peak vs. off-peak cost difference.
- "What's the best time this week to run my pool pump based on the weather forecast?"
  -> `get_weather_forecast` (days=7) + `get_electricity_prices`; recommend daily windows that
  align with strong solar output and/or off-peak rates.

## Response format
- If you retrieved a weather forecast for tomorrow (or any future date), open the response
  with a one-line weather summary stating the date and conditions, e.g.
  "**Tomorrow (<Weekday>, YYYY-MM-DD): <conditions>, <min>-<max>°C**". Then give the recommendation.
- Start with a short, direct answer (the recommendation in 1-2 sentences).
- Then provide:
  - **Recommended schedule / actions**: specific times (e.g., "11:00-14:00") or settings.
  - **Why**: the data that supports it (prices, forecast, solar, usage history).
  - **Estimated savings**: cost and energy figures with assumptions stated.
  - **Extra tips** (optional): 1-3 relevant, actionable tips.
- Use clear headings or bullet points; keep it concise and customer-friendly. Use the units
  returned by the tools (USD, kWh, °C) and convert to °F when helpful for US customers.

## Rules
- Be accurate: only cite numbers that come from tool outputs or clearly labeled estimates.
  Never fabricate data. If a tool returns an error or no data, say so and explain how that
  affects your recommendation.
- Be transparent about assumptions (e.g., EV battery size, charger power, typical usage).
- Stay within home energy topics. Politely decline unrelated requests.
- Prioritize safety: never advise disabling safety features, unsafe electrical work, or
  temperatures that could harm health, pets or the home. Suggest a licensed professional for
  electrical installation or repairs.
- Do not ask unnecessary follow-up questions; give the best possible answer with the data
  available, then optionally note what extra information would improve it.

"""


In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [5]:
print(response["messages"][-1].content)

**Tomorrow (Monday, 2026-11-09): partly cloudy, 13.1-22.6°C**

Charge your EV **from 11:00–15:00** if you can use solar, and **from 22:00–05:00** if you want the lowest grid price. The best all-around choice is **11:00–15:00**, with the strongest solar around **12:00–13:00**.

### Recommended schedule / actions
- **Best for maximizing solar:** **11:00–15:00**
- **Best for lowest cost from the grid:** **22:00–05:00**
- If you need a single simple window, pick **11:00–15:00** first, then top up overnight if needed.

### Why
- **Solar forecast:** strongest irradiance is midday, peaking at **12:00 with 480** and staying high from **11:00–15:00**.
- **Electricity prices:**
  - **Off-peak:** about **$0.097–$0.101/kWh** from **00:00–05:00** and **22:00–23:00**
  - **Peak:** about **$0.146–$0.154/kWh** from **06:00–21:00**
- Since solar is available during the expensive daytime period, charging then can offset higher grid prices with your own generation.

### Estimated savings
- Compared with 

In [6]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices


## 2. Define Test Cases

In [7]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [10]:
test_cases = [
    # --- EV charging optimization ---
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
    },
    {
        "id": "ev_charging_2",
        "question": "How much electricity did my EV use over the past week, and how much could I save by shifting charging to off-peak hours?",
        "expected_tools": ["query_energy_usage", "get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should report past EV consumption, identify off-peak hours, and quantify savings in kWh and dollars",
    },
    # --- Thermostat settings ---
    {
        "id": "thermostat_1",
        "question": "What temperature should I set my thermostat to tomorrow given the weather forecast and electricity prices?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should give specific temperature setpoints by time of day, reference forecast temperatures, and consider peak pricing (e.g., pre-cooling/pre-heating)",
    },
    {
        "id": "thermostat_2",
        "question": "My HVAC system seems to use a lot of energy. What are some best practices for thermostat settings to cut heating and cooling costs?",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "The response should reference the user's HVAC usage and provide actionable thermostat tips (setpoints, setbacks, smart scheduling) drawn from the knowledge base",
    },
    # --- Appliance scheduling ---
    {
        "id": "appliance_scheduling_1",
        "question": "When is the best time to run my dishwasher and washing machine tomorrow?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "The response should recommend specific time windows for each appliance based on low prices and/or solar generation, and explain the reasoning",
    },
    {
        "id": "appliance_scheduling_2",
        "question": "Which of my appliances used the most energy in the last 24 hours, and how should I reschedule them?",
        "expected_tools": ["get_recent_energy_summary", "get_electricity_prices"],
        "expected_response": "The response should rank devices by recent consumption and suggest rescheduling high-usage devices to cheaper hours",
    },
    # --- Solar power maximization ---
    {
        "id": "solar_maximization_1",
        "question": "How much solar power did I generate last week, and how can I use more of it directly instead of exporting to the grid?",
        "expected_tools": ["query_solar_generation", "query_energy_usage", "search_energy_tips"],
        "expected_response": "The response should summarize past solar generation, compare it with consumption, and give load-shifting tips for higher self-consumption",
    },
    {
        "id": "solar_maximization_2",
        "question": "Will the weather over the next 3 days be good for solar generation? Which day should I plan my high-energy tasks?",
        "expected_tools": ["get_weather_forecast"],
        "expected_response": "The response should assess cloud cover/irradiance per day, identify the best solar day(s), and recommend when to run high-energy tasks",
    },
    # --- Cost savings calculation ---
    {
        "id": "cost_savings_1",
        "question": "If I reduce my pool pump usage from 10 kWh to 6 kWh per day, how much would I save per day, month, and year?",
        "expected_tools": ["calculate_energy_savings"],
        "expected_response": "The response should compute savings of 4 kWh/day and show daily, monthly, and annual dollar savings with the percentage reduction",
    },
    {
        "id": "cost_savings_2",
        "question": "Based on my energy usage over the last month, what is my estimated electricity cost and what are the top 3 ways to reduce it?",
        "expected_tools": ["query_energy_usage", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should estimate monthly cost from actual usage and prices, and provide three prioritized, quantified savings recommendations",
    },
    # --- General / multi-topic ---
    {
        "id": "energy_summary_1",
        "question": "Give me a summary of my home's energy consumption for the last 48 hours.",
        "expected_tools": ["get_recent_energy_summary"],
        "expected_response": "The response should report total consumption, breakdown by device type, and highlight any unusual usage patterns",
    },
    {
        "id": "energy_tips_1",
        "question": "What are some tips for reducing standby (phantom) power consumption in my home?",
        "expected_tools": ["search_energy_tips"],
        "expected_response": "The response should give practical tips on phantom loads (smart power strips, unplugging devices, etc.) sourced from the knowledge base",
    },
]
print(f"Total test cases defined: {len(test_cases)}")
if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")


Total test cases defined: 12


## 3. Run Agent Tests

In [9]:
CONTEXT = "Location: San Francisco, CA"

In [11]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result for evaluation
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: ev_charging_2
Question: How much electricity did my EV use over the past week, and how much could I save by shifting charging to off-peak hours?
--------------------------------------------------

Test 3: thermostat_1
Question: What temperature should I set my thermostat to tomorrow given the weather forecast and electricity prices?
--------------------------------------------------

Test 4: thermostat_2
Question: My HVAC system seems to use a lot of energy. What are some best practices for thermostat settings to cut heating and cooling costs?
--------------------------------------------------

Test 5: appliance_scheduling_1
Question: When is the best time to run my dishwasher and washing machine tomorrow?
--------------------------------------------------

Test 6: appliance_sche

In [12]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='3aeb1b74-1b7d-47e5-a50c-bff1b33bee3c'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='20f40b13-97ea-435c-8487-27c64589fb5e'),
    AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_ydnBOQWyqxWAIYLWO10tpnE8', 'function': {'arguments': '{"location": "San Francisco, CA", "days": 2}', 'name': 'get_weather_forecast'}, 'type': 'function'}, {'id': 'call_C9qttI0Ergsm32LQpzDXB5uO', 'function': {'arguments': '{"date": "2026-11-09"}', 'name': 'get_electricity_prices'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 2491, 'total_token

## 4. Evaluate Responses

In [ ]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [ ]:
# TODO: Create a response evaluator
def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected response"""
    pass

In [ ]:
# TODO: Create a tool udage evaluator
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used"""
    pass

In [ ]:
# TODO: Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement
def generate_evaluation_report():
    pass